In [3]:
import os, sys
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine, make_url, text

load_dotenv(find_dotenv(usecwd=True))
url = make_url(os.environ["DATABASE_URL"])
os.environ.update({
    "DBT_HOST": url.host, "DBT_PORT": str(url.port or 5432),
    "DBT_USER": url.username, "DBT_PASSWORD": url.password,
    "DBT_DBNAME": url.database, "DBT_USE_COLORS": "false",
})
engine = create_engine(os.environ["DATABASE_URL"])

PROJECT_DIR = Path.cwd().parent
DBT_DIR = PROJECT_DIR / "fundlens_dbt"
for sub in ["models/staging", "models/core", "snapshots", "macros", "tests"]:
    (DBT_DIR / sub).mkdir(parents=True, exist_ok=True)

exe_dir = Path(sys.executable).parent
DBT_EXE = exe_dir / "dbt.exe" if (exe_dir / "dbt.exe").exists() else exe_dir / "Scripts" / "dbt.exe"
DBT = f"--project-dir {DBT_DIR} --profiles-dir {DBT_DIR}"

print(sys.executable)
!{DBT_EXE} --version

D:\FundLens\venv\Scripts\python.exe
Core:
  - installed: 1.12.5
  - latest:    1.12.5 - Up to date!

Plugins:
  - postgres: 1.11.0 - Up to date!




In [4]:
%%writefile ../fundlens_dbt/dbt_project.yml
name: fundlens
version: "1.0.0"
profile: fundlens

model-paths: ["models"]
snapshot-paths: ["snapshots"]
test-paths: ["tests"]
macro-paths: ["macros"]

models:
  fundlens:
    staging:
      +schema: staging
      +materialized: view
    core:
      +schema: core
      +materialized: table

Writing ../fundlens_dbt/dbt_project.yml


In [5]:
%%writefile ../fundlens_dbt/profiles.yml
fundlens:
  target: dev
  outputs:
    dev:
      type: postgres
      host: "{{ env_var('DBT_HOST') }}"
      port: "{{ env_var('DBT_PORT') | int }}"
      user: "{{ env_var('DBT_USER') }}"
      password: "{{ env_var('DBT_PASSWORD') }}"
      dbname: "{{ env_var('DBT_DBNAME') }}"
      schema: analytics
      threads: 4

Writing ../fundlens_dbt/profiles.yml


In [6]:
%%writefile ../fundlens_dbt/macros/generate_schema_name.sql
{% macro generate_schema_name(custom_schema_name, node) -%}
    {%- if custom_schema_name is none -%}{{ target.schema }}{%- else -%}{{ custom_schema_name | trim }}{%- endif -%}
{%- endmacro %}

Writing ../fundlens_dbt/macros/generate_schema_name.sql


In [7]:
%%writefile ../fundlens_dbt/models/staging/_sources.yml
version: 2

sources:
  - name: raw
    schema: raw
    tables:
      - name: amfi_nav_daily
        description: Daily NAV file from AMFI, all schemes, one row per scheme per day
      - name: nav_history
        description: Historical NAVs from mfapi.in for equity growth schemes
      - name: scheme_meta
        description: Scheme details from mfapi.in
      - name: backfill_status
        description: One row per scheme showing if its history download worked

Writing ../fundlens_dbt/models/staging/_sources.yml


In [8]:
%%writefile ../fundlens_dbt/models/staging/stg_amfi_nav_daily.sql
select
    scheme_code,
    nav_date,
    nav,
    trim(scheme_name)                                   as scheme_name,
    trim(fund_house)                                    as fund_house,
    trim(scheme_category)                               as scheme_category,
    trim(replace(split_part(scheme_category, ' - ', 1), ' Scheme', '')) as asset_class,
    nullif(trim(split_part(scheme_category, ' - ', 2)), '')             as sub_category,
    case when plan ilike 'direct%' then 'Direct'
         when plan ilike 'regular%' then 'Regular' end  as plan_type,
    trim(option)                                        as option_name,
    option ilike '%growth%'                             as is_growth,
    isin_growth,
    loaded_at
from {{ source('raw', 'amfi_nav_daily') }}

Writing ../fundlens_dbt/models/staging/stg_amfi_nav_daily.sql


In [9]:
%%writefile ../fundlens_dbt/models/staging/stg_nav_history.sql
select
    scheme_code,
    nav_date,
    nav,
    loaded_at
from {{ source('raw', 'nav_history') }}
where nav > 0

Writing ../fundlens_dbt/models/staging/stg_nav_history.sql


In [10]:
%%writefile ../fundlens_dbt/models/staging/stg_scheme_latest.sql
-- latest known details for each scheme (input for the SCD2 snapshot)
select distinct on (scheme_code)
    scheme_code, scheme_name, fund_house, scheme_category, asset_class,
    sub_category, plan_type, option_name, is_growth, isin_growth
from {{ ref('stg_amfi_nav_daily') }}
order by scheme_code, nav_date desc

Writing ../fundlens_dbt/models/staging/stg_scheme_latest.sql


In [11]:
%%writefile ../fundlens_dbt/models/staging/_staging.yml
version: 2

models:
  - name: stg_amfi_nav_daily
    columns:
      - name: scheme_code
        data_tests: [not_null]
      - name: nav
        data_tests: [not_null]
  - name: stg_scheme_latest
    columns:
      - name: scheme_code
        data_tests: [unique, not_null]

Writing ../fundlens_dbt/models/staging/_staging.yml


In [12]:
%%writefile ../fundlens_dbt/snapshots/snap_scheme.yml
snapshots:
  - name: snap_scheme
    relation: ref('stg_scheme_latest')
    config:
      schema: snapshots
      unique_key: scheme_code
      strategy: check
      check_cols: [scheme_name, fund_house, scheme_category, plan_type, option_name]

Writing ../fundlens_dbt/snapshots/snap_scheme.yml


In [13]:
%%writefile ../fundlens_dbt/models/core/dim_scheme.sql
-- one row per scheme per version (SCD2). is_current = latest version.
select
    s.dbt_scd_id                       as scheme_sk,
    s.scheme_code,
    s.scheme_name,
    s.fund_house,
    s.scheme_category,
    s.asset_class,
    s.sub_category,
    s.plan_type,
    s.option_name,
    s.is_growth,
    s.isin_growth,
    b.status = 'success'               as has_history,
    b.first_date                       as history_start,
    s.dbt_valid_from                   as valid_from,
    s.dbt_valid_to                     as valid_to,
    s.dbt_valid_to is null             as is_current
from {{ ref('snap_scheme') }} s
left join {{ source('raw', 'backfill_status') }} b using (scheme_code)

Writing ../fundlens_dbt/models/core/dim_scheme.sql


In [14]:
%%writefile ../fundlens_dbt/models/core/fct_nav_daily.sql
{{ config(
    materialized='incremental',
    unique_key=['scheme_code', 'nav_date'],
    incremental_strategy='delete+insert',
    indexes=[{'columns': ['scheme_code', 'nav_date'], 'unique': True}]
) }}

-- one row per scheme per day, only for schemes we track (history downloaded)
with tracked as (
    select scheme_code
    from {{ source('raw', 'backfill_status') }}
    where status = 'success'
),

all_navs as (
    select scheme_code, nav_date, nav, 'amfi' as source, 1 as priority, loaded_at
    from {{ ref('stg_amfi_nav_daily') }}
    where scheme_code in (select scheme_code from tracked)
    union all
    select scheme_code, nav_date, nav, 'mfapi' as source, 2 as priority, loaded_at
    from {{ ref('stg_nav_history') }}
    where scheme_code in (select scheme_code from tracked)
)

{% if is_incremental() %}
-- only rebuild the scheme-days that got new or updated rows since the last run
, changed as (
    select distinct scheme_code, nav_date
    from all_navs
    where loaded_at > (select max(loaded_at) from {{ this }})
)
{% endif %}

select distinct on (a.scheme_code, a.nav_date)
    a.scheme_code,
    a.nav_date,
    a.nav,
    a.source,
    a.loaded_at
from all_navs a
{% if is_incremental() %}
join changed c using (scheme_code, nav_date)
{% endif %}
order by a.scheme_code, a.nav_date, a.priority

Writing ../fundlens_dbt/models/core/fct_nav_daily.sql


In [15]:
%%writefile ../fundlens_dbt/models/core/_core.yml
version: 2

models:
  - name: dim_scheme
    description: Mutual fund schemes with history of name/category changes (SCD2)
    columns:
      - name: scheme_sk
        data_tests: [unique, not_null]
      - name: scheme_code
        data_tests: [not_null]
      - name: plan_type
        data_tests:
          - accepted_values:
              arguments:
                values: ['Direct', 'Regular']
  - name: fct_nav_daily
    description: Daily NAV per tracked scheme (AMFI wins when both sources have the day)
    columns:
      - name: scheme_code
        data_tests:
          - not_null
          - relationships:
              arguments:
                to: ref('dim_scheme')
                field: scheme_code
      - name: nav_date
        data_tests: [not_null]
      - name: nav
        data_tests: [not_null]

Writing ../fundlens_dbt/models/core/_core.yml


In [16]:
tests = {
    "assert_fct_one_row_per_scheme_day.sql": """select scheme_code, nav_date, count(*)
from {{ ref('fct_nav_daily') }}
group by 1, 2
having count(*) > 1""",
    "assert_dim_one_current_row.sql": """select scheme_code, count(*)
from {{ ref('dim_scheme') }}
where is_current
group by 1
having count(*) <> 1""",
    "assert_nav_positive.sql": "select * from {{ ref('fct_nav_daily') }} where nav <= 0",
}
for name, sql in tests.items():
    (DBT_DIR / "tests" / name).write_text(sql)
print(sorted(p.name for p in (DBT_DIR / "tests").iterdir()))

['assert_dim_one_current_row.sql', 'assert_fct_one_row_per_scheme_day.sql', 'assert_nav_positive.sql']


In [17]:
!{DBT_EXE} debug {DBT}

05:51:58  Running with dbt=1.12.5
05:51:58  dbt version: 1.12.5
05:51:58  python version: 3.11.9
05:51:58  python path: D:\FundLens\venv\Scripts\python.exe
05:51:58  os info: Windows-10-10.0.26200-SP0
05:51:58  Using profiles dir at D:\FundLens\fundlens_dbt
05:51:58  Using profiles.yml file at D:\FundLens\fundlens_dbt\profiles.yml
05:51:58  Using dbt_project.yml file at D:\FundLens\fundlens_dbt\dbt_project.yml
05:51:58  adapter type: postgres
05:51:58  adapter version: 1.11.0
05:51:58  Configuration:
05:51:58    profiles.yml file [OK found and valid]
05:51:58    dbt_project.yml file [OK found and valid]
05:51:58  Required dependencies:
05:51:58   - git [OK found]

05:51:58  Connection:
05:51:58    host: localhost
05:51:58    port: 5432
05:51:58    user: postgres
05:51:58    database: fundlens
05:51:58    schema: analytics
05:51:58    connect_timeout: 10
05:51:58    role: None
05:51:58    search_path: None
05:51:58    keepalives_idle: 0
05:51:58    sslmode: None
05:51:58    sslcert: Non

In [18]:
!{DBT_EXE} build {DBT}

05:52:27  Running with dbt=1.12.5
05:52:27  Registered adapter: postgres=1.11.0
05:52:28  Unable to do partial parsing because saved manifest not found. Starting full parse.
05:52:31  Found 5 models, 15 data tests, 1 snapshot, 4 sources, 478 macros
05:52:31  
05:52:31  Concurrency: 4 threads (target='dev')
05:52:31  
05:52:32  1 of 21 START sql view model staging.stg_amfi_nav_daily ........................ [RUN]
05:52:32  2 of 21 START sql view model staging.stg_nav_history ........................... [RUN]
05:52:32  2 of 21 OK created sql view model staging.stg_nav_history ...................... [CREATE VIEW in 0.29s]
05:52:32  1 of 21 OK created sql view model staging.stg_amfi_nav_daily ................... [CREATE VIEW in 0.29s]
05:52:32  3 of 21 START test not_null_stg_amfi_nav_daily_nav ............................. [RUN]
05:52:32  4 of 21 START test not_null_stg_amfi_nav_daily_scheme_code ..................... [RUN]
05:52:32  4 of 21 PASS not_null_stg_amfi_nav_daily_scheme_code ..

In [19]:
display(pd.read_sql(text("""
    select 'dim_scheme' as table_name, count(*) as row_count from core.dim_scheme
    union all
    select 'fct_nav_daily', count(*) from core.fct_nav_daily
"""), engine))

display(pd.read_sql(text("""
    select d.sub_category, d.plan_type, count(distinct f.scheme_code) as schemes,
           min(f.nav_date) as first_date, max(f.nav_date) as last_date
    from core.fct_nav_daily f
    join core.dim_scheme d on d.scheme_code = f.scheme_code and d.is_current
    group by 1, 2
    order by schemes desc
    limit 15
"""), engine))

,table_name,row_count
0,dim_scheme,14136
1,fct_nav_daily,1438372


,sub_category,plan_type,schemes,first_date,last_date
0,Sectoral/ Thematic,Regular,171,2019-01-01,2026-10-12
1,Sectoral/ Thematic,Direct,170,2019-01-01,2026-10-12
2,Thematic Fund,Regular,60,2019-01-01,2026-10-12
3,Thematic Fund,Direct,60,2019-01-01,2026-10-12
4,Flexi Cap Fund,Direct,46,2019-01-01,2026-10-12
5,Flexi Cap Fund,Regular,46,2019-01-01,2026-10-12
6,Large & Mid Cap Fund,Direct,36,2019-01-01,2026-10-12
7,Large & Mid Cap Fund,Regular,36,2019-01-01,2026-10-12
8,Small Cap Fund,Direct,36,2019-01-01,2026-10-12
9,Small Cap Fund,Regular,35,2019-01-01,2026-10-12


In [20]:
!{DBT_EXE} build {DBT}

05:59:47  Running with dbt=1.12.5
05:59:48  Registered adapter: postgres=1.11.0
05:59:49  Found 5 models, 15 data tests, 1 snapshot, 4 sources, 478 macros
05:59:49  
05:59:49  Concurrency: 4 threads (target='dev')
05:59:49  
05:59:50  1 of 21 START sql view model staging.stg_amfi_nav_daily ........................ [RUN]
05:59:50  2 of 21 START sql view model staging.stg_nav_history ........................... [RUN]
05:59:50  2 of 21 OK created sql view model staging.stg_nav_history ...................... [CREATE VIEW in 0.32s]
05:59:50  1 of 21 OK created sql view model staging.stg_amfi_nav_daily ................... [CREATE VIEW in 0.32s]
05:59:50  3 of 21 START test not_null_stg_amfi_nav_daily_nav ............................. [RUN]
05:59:50  4 of 21 START test not_null_stg_amfi_nav_daily_scheme_code ..................... [RUN]
05:59:51  4 of 21 PASS not_null_stg_amfi_nav_daily_scheme_code ........................... [PASS in 0.11s]
05:59:51  3 of 21 PASS not_null_stg_amfi_nav_daily_n